# Joint, Conditional & Bayes

从单变量走向多变量：联合分布描述"一起出现的概率"，条件分布描述"已知一部分后的概率"，贝叶斯定理把它们翻转——这是分类、检测、生成模型的共同数学骨架。


## 0. 环境配置与导入


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import torch
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


PyTorch version: 2.13.0


## 1. 联合分布与边缘分布


联合分布 $p(x, y)$ 描述两个变量同时取值；**边缘分布**是"不管另一个变量"：

$$p(x) = \sum_y p(x, y) \quad(\text{离散}), \qquad p(x) = \int p(x, y)\,dy \quad(\text{连续})$$

边缘化 = 把联合表按另一维求和。


In [2]:
# 离散联合分布 P(X, Y)：X ∈ {0,1}，Y ∈ {0,1,2}
P = np.array([[0.10, 0.15, 0.05],     # X=0 的行
              [0.25, 0.30, 0.15]])    # X=1 的行
print("联合表 P(X,Y)（行 X，列 Y）:")
print(P)
print("行和 = P(X):", P.sum(axis=1))          # 边缘化 Y → P(X)
print("列和 = P(Y):", P.sum(axis=0))          # 边缘化 X → P(Y)
print("总和:", P.sum(), "（应为 1）")


联合表 P(X,Y)（行 X，列 Y）:
[[0.1  0.15 0.05]
 [0.25 0.3  0.15]]
行和 = P(X): [0.3 0.7]
列和 = P(Y): [0.35 0.45 0.2 ]
总和: 1.0 （应为 1）


## 2. 条件概率与条件分布


$$P(A\,|\,B) = \frac{P(A \cap B)}{P(B)}, \qquad p(y\,|\,x) = \frac{p(x, y)}{p(x)}$$

条件分布 = 联合分布"限制在已知事件上再归一化"。独立时 $P(A|B) = P(A)$，即 B 不携带 A 的信息。


In [3]:
# 条件分布：P(Y | X=1)
p_x1 = P[1, :] / P[1, :].sum()
print("P(Y | X=1):", np.round(p_x1, 3))
print("P(Y | X=0):", np.round(P[0, :] / P[0, :].sum(), 3))
print("→ 条件分布随 X 不同而不同，说明 X 携带关于 Y 的信息")


P(Y | X=1): [0.357 0.429 0.214]
P(Y | X=0): [0.333 0.5   0.167]
→ 条件分布随 X 不同而不同，说明 X 携带关于 Y 的信息


## 3. 贝叶斯定理


由条件概率定义直接推出（再配合全概率公式展开分母）：

$$P(A\,|\,B) = \frac{P(B\,|\,A)\,P(A)}{P(B)} = \frac{P(B\,|\,A)\,P(A)}{\sum_A P(B\,|\,A)P(A)}$$

**先验 → 似然 → 后验**：$P(A)$ 是看到数据前的信念，$P(B|A)$ 是数据如何由各假设生成，$P(A|B)$ 是看到数据后的更新。

经典医学检测例子（直觉陷阱）：


In [4]:
prev = 0.01      # 患病率（先验）
sens = 0.99      # 敏感度：P(阳性 | 患病)
spec = 0.95      # 特异度：P(阴性 | 健康)

# P(患病 | 阳性) = sens·prev / (sens·prev + (1-spec)(1-prev))
p_pos = sens*prev + (1-spec)*(1-prev)
post = sens*prev / p_pos
print(f"P(阳性) = {p_pos:.4f}")
print(f"P(患病 | 阳性) = {post:.4f}")
print("→ 即使检测敏感度 99%，阳性者真正患病的概率也只有约 17%：")
print("  因为先验患病率太低，假阳性（5% × 99%）淹没了真阳性")


P(阳性) = 0.0594
P(患病 | 阳性) = 0.1667
→ 即使检测敏感度 99%，阳性者真正患病的概率也只有约 17%：
  因为先验患病率太低，假阳性（5% × 99%）淹没了真阳性


## 4. 朴素贝叶斯分类器


分类任务：根据特征 $f_1,\dots,f_n$ 预测类别 $c$。贝叶斯给出：

$$P(c\,|\,f) \propto P(c)\prod_i P(f_i\,|\,c)$$

"朴素" = 假设特征在给定类别下**条件独立**。虽然假设强，但小数据上很管用。代码：根据天气特征判断要不要出去玩。


In [5]:
# 训练数据：天气特征 [晴天, 冷, 下雨]，标签 [玩, 不玩]
data = [
    ([1, 0, 0], 1), ([1, 1, 0], 0), ([0, 0, 1], 0),
    ([1, 0, 1], 0), ([0, 1, 0], 1), ([1, 1, 1], 0),
    ([0, 1, 1], 0), ([1, 0, 0], 1), ([0, 0, 0], 1),
]
Xs = np.array([d[0] for d in data]); ys = np.array([d[1] for d in data])

def naive_bayes(Xs, ys, x_new, alpha=0.1):
    classes = np.unique(ys)
    scores = {}
    for c in classes:
        mask = ys == c
        prior = mask.mean()
        # 每个特征的条件概率（拉普拉斯平滑）
        cond = [(Xs[mask][:, i].mean() * (1 - alpha) + alpha * 0.5) for i in range(Xs.shape[1])]
        log_score = np.log(prior) + sum(
            np.log(cond[i] if x_new[i] == 1 else 1 - cond[i]) for i in range(len(x_new)))
        scores[c] = log_score
    return scores

x_new = np.array([1, 1, 0])     # 晴天、冷、不下雨
scores = naive_bayes(Xs, ys, x_new)
print("各类别对数后验:", {k: round(v, 3) for k, v in scores.items()})
print("预测:", "玩" if max(scores, key=scores.get) == 1 else "不玩")


各类别对数后验: {np.int64(0): np.float64(-3.113), np.int64(1): np.float64(-2.846)}
预测: 玩


## 5. 生成模型 vs 判别模型的概率视角


- **判别模型**：直接学 $P(y|x)$（逻辑回归、神经网络分类器）——只关心边界
- **生成模型**：学联合分布 $P(x, y)$ 或 $P(x|y)$（朴素贝叶斯、高斯混合、VAE）——能"造数据"

深度学习的分类网络（softmax 输出）本质是判别模型；08 课的 VAE 简析则是生成模型。两者由贝叶斯公式统一：$P(y|x) \propto P(x|y)P(y)$。


## 课后练习


1. **手推贝叶斯**：某疾病患病率 0.001，检测敏感度 0.98、特异度 0.98，求 $P(患病|阳性)$。
2. **全概率公式**：证明 $P(B) = \sum_A P(B|A)P(A)$（把 $B$ 按 $A$ 的取值划分）。
3. **条件独立 ≠ 独立**：给出一个 $X, Y$ 条件独立但整体相关的例子。
4. **朴素贝叶斯**：把 4 节的例子改成"两个特征"重算后验。
5. **思考**：为什么神经网络分类器（判别模型）在小数据上通常优于生成模型？
